# Les choses à nettoyer dans ce dataset #
- Il y'a des rédidus HTML présentes dans la colonne **Condition** à supprimer
- Il y'a 899 valeurs manquantes dans la colonne **Condition** à remplacer par Unknown
- La colonne **date** est de type object et doit être convertie en format date pour PowerBI
- La colonne UniqueID peut être supprimée

In [2]:
import pandas as pd
df = pd.read_csv(r'C:\Users\marie\OneDrive\Documents\GitHub\portfolio-data-ia-pharma\projet1-drug-data\data\drugsComTrain_raw.csv')
print("Informations initiales")
df.shape

Informations initiales


(161297, 7)

### Etape 1 : Les balises HTML dans condition ###

In [3]:
html_mask = df['condition'].str.contains('<', na=False)
print("Lignes avec HTML :", html_mask.sum())
print(df[html_mask]['condition'].value_counts().head(10))

Lignes avec HTML : 900
condition
0</span> users found this comment helpful.    104
2</span> users found this comment helpful.     85
3</span> users found this comment helpful.     84
1</span> users found this comment helpful.     83
4</span> users found this comment helpful.     66
6</span> users found this comment helpful.     46
5</span> users found this comment helpful.     40
7</span> users found this comment helpful.     37
8</span> users found this comment helpful.     31
9</span> users found this comment helpful.     30
Name: count, dtype: int64


In [4]:
import re

df['condition'] = df['condition'].str.replace(r'<[^>]+>', '', regex=True).str.strip()
df['condition'] = df['condition'].replace('', None)  # chaînes vides → NaN
print("Balises restantes :", df['condition'].str.contains('<', na=False).sum())

Balises restantes : 0


### Etape 2 : Valeurs manquantes dans condition ### 

In [5]:
df['condition'].isnull().sum()

np.int64(899)

In [6]:
df['condition'] = df['condition'].fillna('Unknown')
df['condition'].isnull().sum()

np.int64(0)

### Etape 3 : Convertir la colonne date ###

In [7]:
df['date'].describe()

count       161297
unique        3579
top       1-Mar-16
freq           146
Name: date, dtype: object

In [8]:
print(df['date'].dtype)

object


In [9]:
df['date'] = pd.to_datetime(df['date'], format='%d-%b-%y')

In [10]:
print(df['date'].dtype)

datetime64[ns]


In [11]:
df['date'].describe()

count                           161297
mean     2014-06-11 08:51:43.779983360
min                2008-02-24 00:00:00
25%                2012-04-12 00:00:00
50%                2015-06-09 00:00:00
75%                2016-08-19 00:00:00
max                2017-12-12 00:00:00
Name: date, dtype: object

### Etape 4 : Supprimer colonne UniqueID ###

In [12]:
df.columns

Index(['uniqueID', 'drugName', 'condition', 'review', 'rating', 'date',
       'usefulCount'],
      dtype='object')

In [13]:
df = df.drop(columns=['uniqueID'])

In [14]:
df.columns

Index(['drugName', 'condition', 'review', 'rating', 'date', 'usefulCount'], dtype='object')

In [15]:
df.to_csv('../data/drugsComTrain_clean.csv', index=False)

In [16]:
df = pd.read_csv('../data/drugsComTrain_clean.csv')
df.columns
print(df['date'].dtype)

object


In [17]:
df.duplicated().sum()
#donc il n'y a pas de doublons à supprimer

np.int64(0)

In [18]:
df.shape

(161297, 6)

In [19]:
df['date'].describe()

count         161297
unique          3579
top       2016-03-01
freq             146
Name: date, dtype: object

In [20]:
# Charger le CSV propre avec parse_dates
df = pd.read_csv('../data/drugsComTrain_clean.csv', parse_dates=['date'])

# Vérifier le type
print("Type de la colonne date :", df['date'].dtype)

# Statistiques complètes sur la date
print(df['date'].describe())

Type de la colonne date : datetime64[ns]
count                           161297
mean     2014-06-11 08:51:43.779983360
min                2008-02-24 00:00:00
25%                2012-04-12 00:00:00
50%                2015-06-09 00:00:00
75%                2016-08-19 00:00:00
max                2017-12-12 00:00:00
Name: date, dtype: object


Le dataset couvre la période 2008-2017, avec une concentration  des avis entre 2012 et 2016 (50% des données dans cet intervalle).